# 🎵 VoksplitNet : Entraînement 100% Commercial & Export ONNX FP16

Ce notebook vous permet d'entraîner le modèle d'isolation vocale propriétaire **VoksplitNet** (TFC-TDF U-Net) en utilisant le **Dynamic Synthetic Remixing** (100% conforme aux usages commerciaux, libre de toute restriction NC).

### ⚖️ Approche Commerciale (100% Libre de Droits) :
- **Acapellas Libres** : CC0 / CC-BY (ccMixter, VocalSet, Mozilla Common Voice)
- **Instrumentaux Libres** : CC0 / CC-BY (Slakh2100, Free Music Archive CC0/CC-BY)
- **Zéro MUSDB18-HQ / Zéro Poids Meta HTDemucs** : aucune clause Non-Commerciale (NC).
- **Modèle Final** : ONNX FP16 (~16 Mo) prêt pour WebGPU et WASM multithread local.

### ⚡ Prérequis Colab :
1. Allez dans le menu **Exécution** > **Modifier le type d'exécution**.
2. Sélectionnez **T4 GPU** (accélérateur matériel gratuit) et enregistrez.
3. Exécutez les cellules pas à pas ou cliquez sur **Tout exécuter**.

In [ ]:
# 1. Installation des dépendances audio, ONNX et soundfile
!pip install -q torchaudio onnx onnxruntime onnxconverter-common onnxscript soundfile matplotlib


In [ ]:
# 2. Vérification de l'accélération GPU
import torch
print(f"PyTorch Version : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU détecté : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Aucun GPU détecté. Pensez à activer le T4 GPU dans Exécution > Modifier le type d'exécution.")

In [ ]:
# 3. Préparation des Dossiers Commerciaux (Dynamic Remixing) & Décompression Automatique
import os
import glob
import zipfile

DATA_DIR = "./data/commercial_stems"
voc_train = f"{DATA_DIR}/train/vocals"
ins_train = f"{DATA_DIR}/train/instrumentals"
os.makedirs(voc_train, exist_ok=True)
os.makedirs(ins_train, exist_ok=True)

# Détection et décompression automatique des archives .zip déposées dans Colab
for item in os.listdir("/content"):
    if item.endswith(".zip"):
        zpath = os.path.join("/content", item)
        print(f"📦 Archive détectée : {item} -> Décompression automatique...")
        with zipfile.ZipFile(zpath, "r") as zf:
            namelist = [n for n in zf.namelist() if not n.startswith("__MACOSX") and not os.path.basename(n).startswith(".") and not n.endswith("/")]
            # Si archive vocale
            if "vocal" in item.lower() or any("f1_" in n or "m1_" in n or "row_" in n for n in namelist):
                for member in namelist:
                    if member.lower().endswith((".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus")):
                        filename = os.path.basename(member)
                        with zf.open(member) as source, open(os.path.join(voc_train, filename), "wb") as target:
                            target.write(source.read())
                print(f"   ✅ Voix extraites avec succès vers {voc_train}")
            # Si archive instrumentale
            elif "instru" in item.lower() or any(".opus" in n or ".mp3" in n for n in namelist):
                for member in namelist:
                    if member.lower().endswith((".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus")):
                        filename = os.path.basename(member)
                        with zf.open(member) as source, open(os.path.join(ins_train, filename), "wb") as target:
                            target.write(source.read())
                print(f"   ✅ Instrumentaux extraits avec succès vers {ins_train}")

num_v = len([f for f in os.listdir(voc_train) if not f.startswith(".")])
num_i = len([f for f in os.listdir(ins_train) if not f.startswith(".")])
print(f"\n📁 Dossiers prêts pour VoksplitNet v2 :")
print(f"   - Acapellas prêts : {num_v} fichiers dans {voc_train}")
print(f"   - Instrumentaux prêts : {num_i} fichiers dans {ins_train}")


In [ ]:
# 4. Définition de l'architecture VoksplitNet (TFC-TDF U-Net)
import torch.nn as nn
import torch.nn.functional as F

class TFC(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.GELU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.shortcut(x)
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.act2(self.bn2(self.conv2(x)))
        return x + res

class TDF(nn.Module):
    def __init__(self, channels, num_freq_bins, bottleneck_ratio=4):
        super().__init__()
        bottleneck = max(8, num_freq_bins // bottleneck_ratio)
        self.linear1 = nn.Linear(num_freq_bins, bottleneck)
        self.act = nn.GELU()
        self.linear2 = nn.Linear(bottleneck, num_freq_bins)
        self.norm = nn.LayerNorm(num_freq_bins)

    def forward(self, x):
        b, c, f, t = x.shape
        x_trans = x.permute(0, 1, 3, 2)
        res = x_trans
        out = self.linear2(self.act(self.linear1(x_trans)))
        out = self.norm(out + res)
        return out.permute(0, 1, 3, 2)

class TFC_TDF_Block(nn.Module):
    def __init__(self, in_channels, out_channels, num_freq_bins):
        super().__init__()
        self.tfc = TFC(in_channels, out_channels)
        self.tdf = TDF(out_channels, num_freq_bins)

    def forward(self, x):
        return self.tdf(self.tfc(x))

class VoksplitNet(nn.Module):
    def __init__(self, in_channels=2, base_channels=32, num_freq_bins=1024):
        super().__init__()
        self.num_freq_bins = num_freq_bins
        self.enc1 = TFC_TDF_Block(in_channels, base_channels, num_freq_bins)
        self.down1 = nn.MaxPool2d(kernel_size=(2, 2))
        f_down1 = num_freq_bins // 2
        self.enc2 = TFC_TDF_Block(base_channels, base_channels * 2, f_down1)
        self.down2 = nn.MaxPool2d(kernel_size=(2, 2))
        f_down2 = f_down1 // 2
        self.enc3 = TFC_TDF_Block(base_channels * 2, base_channels * 4, f_down2)
        self.down3 = nn.MaxPool2d(kernel_size=(2, 2))
        f_down3 = f_down2 // 2
        self.bottleneck = TFC_TDF_Block(base_channels * 4, base_channels * 8, f_down3)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=(2, 2), stride=(2, 2))
        self.dec3 = TFC_TDF_Block(base_channels * 8, base_channels * 4, f_down2)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=(2, 2), stride=(2, 2))
        self.dec2 = TFC_TDF_Block(base_channels * 4, base_channels * 2, f_down1)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=(2, 2), stride=(2, 2))
        self.dec1 = TFC_TDF_Block(base_channels * 2, base_channels, num_freq_bins)
        self.head_vocals = nn.Sequential(nn.Conv2d(base_channels, 2, kernel_size=1), nn.Sigmoid())
        self.head_instrumental = nn.Sequential(nn.Conv2d(base_channels, 2, kernel_size=1), nn.Sigmoid())

    def forward(self, x):
        orig_t = x.shape[3]
        pad_t = (8 - (orig_t % 8)) % 8
        if pad_t > 0:
            x = F.pad(x, (0, pad_t, 0, 0))
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        b = self.bottleneck(self.down3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        mask_voc = self.head_vocals(d1)[:, :, :, :orig_t]
        mask_ins = self.head_instrumental(d1)[:, :, :, :orig_t]
        return mask_voc, mask_ins

print("Modèle VoksplitNet instancié avec succès.")

In [ ]:
# 5. Dataset Commercial V2 (Ultra-Rapide) & Boucle d'Entraînement Studio
import os
import glob
import random
import time
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio

# Nettoyage immédiat de la VRAM GPU
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

class CommercialStemDataset(Dataset):
    def __init__(self, root_dir, split="train", sample_rate=44100, segment_seconds=6.0, n_fft=2048, hop_length=512, augment=True):
        super().__init__()
        self.sample_rate = sample_rate
        self.segment_seconds = segment_seconds
        self.segment_samples = int(segment_seconds * sample_rate)
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.augment = augment and (split == "train")
        self.window = torch.hann_window(n_fft)
        
        valid_exts = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".opus", ".aif", ".aiff")
        voc_dir = os.path.join(root_dir, split, "vocals") if os.path.exists(os.path.join(root_dir, split, "vocals")) else os.path.join(root_dir, "vocals")
        ins_dir = os.path.join(root_dir, split, "instrumentals") if os.path.exists(os.path.join(root_dir, split, "instrumentals")) else os.path.join(root_dir, "instrumentals")
        
        self.vocal_files = []
        self.instrumental_files = []
        
        if os.path.exists(voc_dir):
            for root, _, files in os.walk(voc_dir):
                for f in files:
                    if not f.startswith(".") and f.lower().endswith(valid_exts):
                        self.vocal_files.append(os.path.join(root, f))
                        
        if os.path.exists(ins_dir):
            for root, _, files in os.walk(ins_dir):
                for f in files:
                    if not f.startswith(".") and f.lower().endswith(valid_exts):
                        self.instrumental_files.append(os.path.join(root, f))
                        
        self.vocal_files = sorted(self.vocal_files)
        self.instrumental_files = sorted(self.instrumental_files)
        
        print(f"✅ [{split.upper()}] Pistes détectées : {len(self.vocal_files)} Acapellas | {len(self.instrumental_files)} Instrumentaux")

    def __len__(self):
        if len(self.vocal_files) == 0 or len(self.instrumental_files) == 0: return 50
        return max(len(self.vocal_files), len(self.instrumental_files)) * 8

    def _load_audio_chunk(self, path):
        try:
            # 1. Chargement du flux audio
            wf, sr = torchaudio.load(path)
            if wf.shape[0] == 1:
                wf = wf.repeat(2, 1)
            elif wf.shape[0] > 2:
                wf = wf[:2, :]
            
            # 2. Découper d'abord le petit extrait de 6s (40x plus rapide en mémoire)
            total_samples = wf.shape[-1]
            target_raw = int(self.segment_seconds * sr)
            if total_samples > target_raw:
                max_offset = total_samples - target_raw
                offset = random.randint(0, max_offset) if self.augment else 0
                wf = wf[:, offset:offset + target_raw]
            elif total_samples < target_raw:
                wf = F.pad(wf, (0, target_raw - total_samples))
            
            # 3. Rééchantillonner UNIQUEMENT le segment de 6s vers 44.1 kHz (instantané)
            if sr != self.sample_rate:
                wf = torchaudio.functional.resample(wf, sr, self.sample_rate)
            
            # 4. Ajuster précisément à 263680 échantillons
            if wf.shape[-1] > self.segment_samples:
                wf = wf[:, :self.segment_samples]
            elif wf.shape[-1] < self.segment_samples:
                wf = F.pad(wf, (0, self.segment_samples - wf.shape[-1]))
            return wf
        except Exception as e:
            return torch.zeros(2, self.segment_samples)

    def _compute_stft_magnitude(self, wf):
        stft = torch.stft(wf, n_fft=self.n_fft, hop_length=self.hop_length, window=self.window.to(wf.device), return_complex=True)
        return torch.abs(stft)[:, :1024, :]

    def __getitem__(self, idx):
        if len(self.vocal_files) == 0 or len(self.instrumental_files) == 0:
            mix = torch.rand(2, 1024, 256) * 2.0
            voc = mix * torch.rand(2, 1024, 256)
            ins = torch.clamp(mix - voc, min=0.0)
            return mix, voc, ins
        
        voc_wf = self._load_audio_chunk(random.choice(self.vocal_files))
        ins_wf = self._load_audio_chunk(random.choice(self.instrumental_files))
        
        # Augmentation dynamique de studio
        if self.augment:
            voc_wf = voc_wf * random.uniform(0.5, 1.4)
            ins_wf = ins_wf * random.uniform(0.6, 1.3)
            if random.random() < 0.5:
                voc_wf = torch.flip(voc_wf, dims=[0])
                ins_wf = torch.flip(ins_wf, dims=[0])
                
        mix_wf = voc_wf + ins_wf
        peak = torch.max(torch.abs(mix_wf)).item()
        if peak > 0.95:
            scale = 0.95 / peak
            mix_wf, voc_wf, ins_wf = mix_wf * scale, voc_wf * scale, ins_wf * scale
            
        return self._compute_stft_magnitude(mix_wf), self._compute_stft_magnitude(voc_wf), self._compute_stft_magnitude(ins_wf)

# ==========================================================
# 🎛️ PARAMÈTRES D'ENTRAÎNEMENT V2 (STUDIO COMMERCIAL)
# ==========================================================
batch_size = 4             # Optimisé pour la mémoire du GPU T4
num_epochs = 40            # 40 époques d'apprentissage
learning_rate = 8e-4       # Vitesse d'apprentissage adaptative
max_batches_per_epoch = 60 # 60 batchs x 4 = 240 remix par époque (~15-20s par époque)
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VoksplitNet().to(device)
dataset = CommercialStemDataset(DATA_DIR, split="train")
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=(device.type == "cuda"))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

print(f"🔥 Lancement de l'entraînement VoksplitNet V2 sur {device}...")
print(f"   - Batch size : {batch_size} | Époques : {num_epochs} | LR initial : {learning_rate}")
print(f"   - Échantillons par époque : ~{max_batches_per_epoch * batch_size} remix dynamiques")
print(f"   - Durée estimée totale : ~12 à 15 minutes sur T4 GPU")

eps = 1e-6
for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    epoch_voc_loss = 0.0
    epoch_ins_loss = 0.0
    t0 = time.time()
    
    for b_idx, (mix_mag, voc_mag, ins_mag) in enumerate(loader):
        mix_mag, voc_mag, ins_mag = mix_mag.to(device), voc_mag.to(device), ins_mag.to(device)
        optimizer.zero_grad()
        mask_voc, mask_ins = model(mix_mag)
        est_voc = mask_voc * mix_mag
        est_ins = mask_ins * mix_mag
        
        l1_voc = F.l1_loss(est_voc, voc_mag)
        l1_ins = F.l1_loss(est_ins, ins_mag)
        log_voc = F.l1_loss(torch.log(est_voc + eps), torch.log(voc_mag + eps))
        log_ins = F.l1_loss(torch.log(est_ins + eps), torch.log(ins_mag + eps))
        mask_sum_loss = F.mse_loss(mask_voc + mask_ins, torch.ones_like(mask_voc))
        
        loss = (l1_voc + l1_ins) + 0.12 * (log_voc + log_ins) + 0.05 * mask_sum_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_voc_loss += l1_voc.item()
        epoch_ins_loss += l1_ins.item()
        
        # Progression en temps réel toutes les 5 itérations
        if (b_idx + 1) % 5 == 0 or (b_idx + 1) == max_batches_per_epoch:
            pct = int(((b_idx + 1) / max_batches_per_epoch) * 100)
            print(f"  [Époque {epoch:02d}/{num_epochs}] Batch {b_idx+1:02d}/{max_batches_per_epoch} ({pct:3d}%) | Perte : {loss.item():.4f} (Voc: {l1_voc.item():.4f}, Ins: {l1_ins.item():.4f})", end="\r")
        
        if b_idx + 1 >= max_batches_per_epoch:
            break
            
    scheduler.step()
    batches_done = min(len(loader), max_batches_per_epoch)
    avg_loss = epoch_loss / batches_done
    avg_voc = epoch_voc_loss / batches_done
    avg_ins = epoch_ins_loss / batches_done
    elapsed = time.time() - t0
    print(f"  Époque {epoch:02d}/{num_epochs} - Perte : {avg_loss:.4f} (Voc L1: {avg_voc:.4f}, Ins L1: {avg_ins:.4f}) - Temps : {elapsed:.1f}s")

print("\n🏆 Entraînement VoksplitNet V2 terminé avec succès ! Prêt pour l'export ONNX FP16.")


In [ ]:
# 6. Export ONNX standard (FP32) & Optimisation WebGPU (FP16)
import os
import sys
import hashlib
import subprocess

# Vérification et auto-installation d'onnxscript et onnxconverter-common
try:
    import onnx
    import onnxscript
    from onnxconverter_common import float16
except ImportError:
    print("Installation automatique des dépendances ONNX manquantes (onnxscript, etc.)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "onnxscript"])
    import onnx
    import onnxscript
    from onnxconverter_common import float16

os.makedirs("./exported_models", exist_ok=True)
model.eval().to("cpu")

dummy_input = torch.randn(1, 2, 1024, 512)
fp32_path = "./exported_models/voksplit-net.onnx"
fp16_path = "./exported_models/voksplit-net-fp16.onnx"

print("Export en cours vers ONNX...")
try:
    torch.onnx.export(
        model,
        dummy_input,
        fp32_path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["mix_mag"],
        output_names=["mask_vocals", "mask_instrumental"],
        dynamic_axes={
            "mix_mag": {0: "batch_size", 3: "time_frames"},
            "mask_vocals": {0: "batch_size", 3: "time_frames"},
            "mask_instrumental": {0: "batch_size", 3: "time_frames"}
        }
    )
except ModuleNotFoundError as e:
    if "onnxscript" in str(e):
        print("Module onnxscript manquant détecté. Installation en cours...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxscript"])
        torch.onnx.export(
            model,
            dummy_input,
            fp32_path,
            export_params=True,
            opset_version=17,
            do_constant_folding=True,
            input_names=["mix_mag"],
            output_names=["mask_vocals", "mask_instrumental"],
            dynamic_axes={
                "mix_mag": {0: "batch_size", 3: "time_frames"},
                "mask_vocals": {0: "batch_size", 3: "time_frames"},
                "mask_instrumental": {0: "batch_size", 3: "time_frames"}
            }
        )
    else:
        raise e

print(f"✅ Modèle ONNX FP32 créé : {fp32_path} ({os.path.getsize(fp32_path)/(1024*1024):.2f} Mo)")

# Optimisation FP16 WebGPU avec secours sécurisé
try:
    onnx_m = onnx.load(fp32_path)
    onnx_fp16 = float16.convert_float_to_float16(onnx_m, keep_io_types=True)
    onnx.save(onnx_fp16, fp16_path)
    size_mb = os.path.getsize(fp16_path) / (1024 * 1024)
    print(f"✅ Modèle FP16 optimisé généré : {fp16_path} ({size_mb:.2f} Mo)")
    target_file = fp16_path
except Exception as e:
    print(f"⚠️ Conversion FP16 contournée ({e}). Utilisation directe du modèle FP32.")
    target_file = fp32_path

# Calcul de la signature SHA-256 pour l'intégrité Web
sha = hashlib.sha256()
with open(target_file, "rb") as f:
    while chunk := f.read(65536):
        sha.update(chunk)
hash_str = sha.hexdigest()
print(f"🔑 Signature SHA-256 : {hash_str}")


In [ ]:
# 7. Téléchargement immédiat du modèle optimisé sur votre machine
try:
    from google.colab import files
    print(f"Téléchargement du fichier {target_file}...")
    files.download(target_file)
except Exception as e:
    print(f"📁 Si le téléchargement automatique est bloqué, faites simplement un clic-droit sur '{target_file}' dans l'explorateur de fichiers à gauche > Télécharger.")